# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library. All dataset components are referenced by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, columns, and corresponding Croissant `@id` values for reference.

In [ ]:
# List all available record sets and their IDs
print('\nAvailable record sets (@id):')
record_sets = list(dataset.record_sets)
for rset in record_sets:
    recset_obj = dataset.record_sets[rset]
    print(f"- @id: {recset_obj.id}")
    print(f"  name: {recset_obj.name if hasattr(recset_obj, 'name') else '(no name specified)'}")
    # Print fields within this record set (by @id)
    print('  fields:')
    for fld in getattr(recset_obj, 'fields', []):
        print(f"      - @id: {fld.id} (name: {fld.name if hasattr(fld, 'name') else '(no name specified)'})")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- Use the record set and field `@id`s identified in the previous section. All accesses should reference the canonical Croissant `@id`.

In [ ]:
# Extract data for every record set
dataframes = {}
for recset_id, recset_obj in dataset.record_sets.items():
    print(f"Loading records for record set @id: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  Loaded {len(df)} rows and {df.shape[1]} columns\n")
    if len(dataframes) == 1:
        # Print columns and preview only for first one
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
        first_recset_id = recset_id

## 4. Exploratory Data Analysis (EDA)
Apply basic exploration: filter records, normalize numeric fields, and group data by key attributes.

**Note:** Adjust the `numeric_field_id` and `group_field_id` in the code below according to the schema overview above (use the `@id` of your chosen fields).

In [ ]:
# For demonstration: auto-select a numeric field by type if schema provided such info
df = dataframes[first_recset_id]

# Try to determine a likely numeric field
numeric_field_id = None
for recset in dataset.record_sets.values():
    for field in getattr(recset, 'fields', []):
        # Try common numeric field types/ids
        if hasattr(field, 'data_type') and field.data_type in ('Float', 'Integer', 'Number'):
            if field.id in df.columns:
                numeric_field_id = field.id
                break
    if numeric_field_id:
        break

if not numeric_field_id:
    # Fallback: select the first numeric-looking column
    numcols = df.select_dtypes(include=['number']).columns
    if len(numcols):
        numeric_field_id = numcols[0]
    else:
        raise ValueError("No numeric columns found for EDA.")

print(f"Using numeric field: {numeric_field_id}")

# Define a threshold for filtering
threshold = df[numeric_field_id].quantile(0.80)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 20%): {len(filtered_df)} rows")

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to select a group field (categorical)
group_field_id = None
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() < len(df) // 4:
        group_field_id = col
        break

if group_field_id:
    print(f"Grouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll plot the distribution of the selected numeric field and (if available) the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id], kde=True, bins=10, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped_df exists, plot grouped means
if 'grouped_df' in locals() and not grouped_df.empty:
    grouped_df.plot(kind='bar', figsize=(10, 4), legend=False)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

Using `mlcroissant`, we have:
- Loaded metadata and records from the FAIR² colorectal cancer survivor dataset via Croissant schema URL
- Explored its record sets, fields, and columns using their canonical `@id` values
- Performed basic EDA: numeric field filtering, normalization, groupwise means, and distribution visualizations

**Next steps:** You can further analyze or model this dataset by exploring additional record sets or processing new fields—always using their Croissant `@id`.

_For more information on `mlcroissant`, see: https://mlcommons.github.io/croissant/api/mlcroissant/_autosummary/mlcroissant.Dataset.html_
